# VisionDecode — Model Zoo (Models 1–8)

Distorted-text OCR scored by **Character Error Rate (CER)**. This notebook implements and
compares every model in the project plan. Model 1 (EfficientNet-B0 multi-head baseline) is
carried over from `prototype_model_2.ipynb`; Models 2–8 are built on top of shared
infrastructure so results are directly comparable.

| # | Model | Backbone | Sequence head | Notes |
|---|-------|----------|---------------|-------|
| 1 | Baseline | EfficientNet-B0 | 6 independent linear heads | reference |
| 2 | CRNN | EfficientNet-B0 | BiGRU ×2 (h=256) | industry standard |
| 3 | Lightweight | MobileNetV3-Small | BiGRU ×2 (h=128) | fast / small |
| 4 | ViT | vit_small_patch16_224 (timm) | 6 heads on pooled token | transformer |
| 5 | Tiny CNN | custom (<500k params) | 6 shared head | real-time CPU |
| 6 | Ensemble | top-3 of the above | per-position vote / avg | hybrid |
| 7 | ResNet | resNet |  gru | reference|

> **Charset note.** The task brief mentions "31 classes," but the actual training labels use a
> **33-character** alphabet (`0-9` + `A-Z` minus the ambiguous `I`, `L`, `O`). All classification
> heads below output **33** classes; the CTC variant adds a blank (34). Two malformed CSV rows
> (lengths 8–9, lowercase/punctuation) are filtered out on load.

> **Training is intentionally left unexecuted** — it is time-consuming. Each model has a build
> cell (safe to run) and a training cell (run on a GPU to reproduce).

In [1]:
import os
import glob
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
print('Using device:', device)

Using device: cuda


## Shared infrastructure

Charset, encoding, the CER metric, a dataset, a loader factory, and a reusable
classification-style training / evaluation loop. Every classification model (1, 2, 3, 4, 6)
predicts a fixed `6 × 33` grid and reuses these helpers; TrOCR (5) and the traditional baselines
(8) have their own pipelines.

In [2]:
# --- Charset (33 classes; no I/L/O) ---
CHARS = '0123456789ABCDEFGHJKMNPQRSTUVWXYZ'
NUM_CLASSES = len(CHARS)        # 33  (classification heads)
CODE_LEN = 6

char_to_idx = {c: i for i, c in enumerate(CHARS)}
idx_to_char = {i: c for i, c in enumerate(CHARS)}

def encode(text):
    return torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

def decode(indices):
    return ''.join(idx_to_char[int(i)] for i in indices)

print('Classification classes:', NUM_CLASSES, '| code length:', CODE_LEN)

Classification classes: 33 | code length: 6


In [3]:
# --- Character Error Rate (CER) via Levenshtein distance ---
def levenshtein(a, b):
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, m + 1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
            prev = cur
    return dp[m]

def cer(preds, targets):
    edits = sum(levenshtein(p, t) for p, t in zip(preds, targets))
    chars = sum(len(t) for t in targets)
    return edits / max(chars, 1)

In [4]:
# --- Dataset (returns RGB image + 6-length label tensor) ---
DATA_DIR = '../data'

class CaptchaDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        df = pd.read_csv(csv_file)
        df = df[df['text'].astype(str).str.len() == CODE_LEN].reset_index(drop=True)
        df = df[df['text'].apply(lambda t: all(c in char_to_idx for c in str(t)))].reset_index(drop=True)
        self.df, self.img_dir, self.transform = df, img_dir, transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['image'])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, encode(row['text'])

def make_loaders(transform, batch_size=64, val_frac=0.1, seed=42):
    full = CaptchaDataset(os.path.join(DATA_DIR, 'train-labels.csv'),
                          os.path.join(DATA_DIR, 'train_images'), transform)
    val_size = int(val_frac * len(full))
    train, val = random_split(full, [len(full) - val_size, val_size],
                              generator=torch.Generator().manual_seed(seed))
    tl = DataLoader(train, batch_size=batch_size, shuffle=True,  num_workers=0)
    vl = DataLoader(val,   batch_size=128,        shuffle=False, num_workers=0)
    return tl, vl

In [5]:
# --- Shared classification train / eval (output shape: (B, 6, 33)) ---
def decode_logits(logits):
    idx = logits.argmax(dim=2).cpu()
    return [decode(row) for row in idx]

@torch.no_grad()
def evaluate_cls(model, loader):
    model.eval()
    preds, trues, seq_ok, seq_n = [], [], 0, 0
    for images, labels in loader:
        out = model(images.to(device))            # (B, 6, 33)
        p = out.argmax(dim=2).cpu()
        for pi, ti in zip(p, labels):
            preds.append(decode(pi)); trues.append(decode(ti))
        seq_ok += (p == labels).all(dim=1).sum().item()
        seq_n  += labels.size(0)
    return cer(preds, trues), seq_ok / max(seq_n, 1)

def cls_loss(logits, labels):
    return sum(F.cross_entropy(logits[:, p, :], labels[:, p]) for p in range(CODE_LEN))

def train_classifier(model, train_loader, val_loader, epochs=15, lr=1e-3, ckpt=None):
    model = model.to(device)
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=2)
    best = float('inf')
    if ckpt:
        os.makedirs(os.path.dirname(ckpt), exist_ok=True)
    for epoch in range(1, epochs + 1):
        model.train(); running = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            opt.zero_grad()
            loss = cls_loss(model(images), labels)
            loss.backward(); opt.step()
            running += loss.item() * images.size(0)
        tr = running / len(train_loader.dataset)
        vcer, sacc = evaluate_cls(model, val_loader); sched.step(vcer)
        print(f'Epoch {epoch:2d} | loss {tr:.4f} | val CER {vcer:.4f} | full-code acc {sacc:.4f}')
        if ckpt and vcer < best:
            best = vcer; torch.save(model.state_dict(), ckpt)
            print(f'  ↳ best CER {best:.4f} saved -> {ckpt}')
    return best

# Default ImageNet transform for the pretrained CNN backbones (Models 1-3, 6).
imagenet_tf = transforms.Compose([
    transforms.Resize((100, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## Model 1 — Baseline: EfficientNet-B0 + 6 multi-heads *(reference)*

Carried over from `prototype_model_2.ipynb`. A frozen EfficientNet-B0 trunk (last 3 blocks
fine-tuned) feeds 6 independent classification heads — one per character position. Serves as the
comparison anchor for everything below.

In [6]:
class CaptchaEfficientNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN):
        super().__init__()
        self.code_len = code_len
        self.model = models.efficientnet_b0(weights='DEFAULT')
        for p in self.model.parameters():
            p.requires_grad = False
        for p in self.model.features[-3:].parameters():
            p.requires_grad = True
        in_features = self.model.classifier[1].in_features
        self.classifiers = nn.ModuleList([
            nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, 256), nn.ReLU(),
                          nn.Dropout(0.3), nn.Linear(256, num_classes))
            for _ in range(code_len)
        ])

    def forward(self, x):
        f = self.model.features(x)
        f = torch.flatten(self.model.avgpool(f), 1)
        return torch.stack([h(f) for h in self.classifiers], dim=1)  # (B, 6, 33)

model = CaptchaEfficientNet()
print('Model 1 params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Model 1 params: 5174242


In [ ]:
# --- Train Model 1 (run on GPU) ---
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model, train_loader, val_loader, epochs=15, lr=1e-3,
                 ckpt='../Artifacts/model1_efficientnet.pth')

In [8]:
train_loader, val_loader = make_loaders(imagenet_tf)
model = CaptchaEfficientNet().to(device)
model.load_state_dict(torch.load('../Artifacts/model1_efficientnet.pth', map_location=device))
vcer, sacc = evaluate_cls(model, val_loader)
print('Model 1 val CER:', vcer, '| full-code acc:', sacc)

Model 1 val CER: 0.09763214940803736 | full-code acc: 0.527263631815908


## Model 2 — CRNN: EfficientNet-B0 + BiGRU (industry standard)

The EfficientNet feature map is pooled to a width-6 sequence (`AdaptiveAvgPool2d((1, 6))`),
giving 6 timesteps that a 2-layer **bidirectional GRU** (hidden 256) reads left-to-right and
right-to-left before a shared linear layer emits 33 classes per timestep.

*Alternative:* keep the full feature width and train with **CTC loss** for true variable-length
decoding — see `crnn_ctc_model.ipynb` for that variant. Here we use fixed 6-step CE so it stays
directly comparable with the other classification models.

In [10]:
class CRNN_EfficientNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=256):
        super().__init__()
        self.code_len = code_len
        backbone = models.efficientnet_b0(weights='DEFAULT')
        for p in backbone.parameters():
            p.requires_grad = False
        for p in backbone.features[-3:].parameters():
            p.requires_grad = True
        self.features = backbone.features          # output channels: 1280
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))   # -> (B, 1280, 1, 6)
        self.gru = nn.GRU(1280, hidden, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        f = self.features(x)                       # (B, 1280, H, W)
        f = self.pool(f).squeeze(2)                # (B, 1280, 6)
        f = f.permute(0, 2, 1)                     # (B, 6, 1280)
        seq, _ = self.gru(f)                       # (B, 6, 2*hidden)
        return self.fc(seq)                        # (B, 6, 33)

model = CRNN_EfficientNet()
print('Model params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Model params: 6717757


In [ ]:
# --- Train Model 2 (run on GPU) ---
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model, train_loader, val_loader, epochs=20, lr=1e-3,
                 ckpt='../Artifacts/model2_crnn_bigru.pth')

In [11]:
model = CRNN_EfficientNet().to(device)
model.load_state_dict(torch.load('../Artifacts/model2_crnn_bigru.pth', map_location=device))
vcer, sacc = evaluate_cls(model, val_loader)
print('Model 2 val CER:', vcer, '| full-code acc:', sacc)

Model 2 val CER: 0.07270301817575454 | full-code acc: 0.6373186593296648


## Model 3 — Lightweight: MobileNetV3-Small + BiGRU

Same CRNN recipe as Model 2 but with a **MobileNetV3-Small** trunk (576 feature channels) and a
smaller GRU (hidden 128). Targets fast inference and a small footprint while keeping the sequence
head.

In [12]:
class CRNN_MobileNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=128):
        super().__init__()
        self.code_len = code_len
        backbone = models.mobilenet_v3_small(weights='DEFAULT')
        self.features = backbone.features          # output channels: 576
        feat_ch = 576
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))
        self.gru = nn.GRU(feat_ch, hidden, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        f = self.features(x)                       # (B, 576, H, W)
        f = self.pool(f).squeeze(2).permute(0, 2, 1)  # (B, 6, 576)
        seq, _ = self.gru(f)
        return self.fc(seq)                        # (B, 6, 33)

model = CRNN_MobileNet()
print('Model  params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Model  params: 1774145


In [ ]:
# --- Train Model 3 (run on GPU) ---
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model, train_loader, val_loader, epochs=20, lr=1e-3,
                 ckpt='../Artifacts/model3_mobilenet_gru.pth')

In [13]:
model = CRNN_MobileNet().to(device)
model.load_state_dict(torch.load('../Artifacts/model3_mobilenet_gru.pth', map_location=device))
vcer, sacc = evaluate_cls(model, val_loader)
print('Model 3 val CER:', vcer, '| full-code acc:', sacc)

Model 3 val CER: 0.016091379022844757 | full-code acc: 0.9069534767383692


## Model 4 — Vision Transformer (timm `vit_small_patch16_224`)

A pretrained ViT-Small encodes the image (resized to 224×224) into a pooled embedding; 6
classification heads then predict the 6 characters. Uses the `timm` library.

*Notes:* DeiT (`deit_small_patch16_224`) is a drop-in `model_name` swap and is more
data-efficient. To get a true *sequence* output instead of pooled features, set
`num_classes=0, global_pool=''` and attach a head to the patch tokens — kept simple (pooled +
6 heads) here for comparability.

In [14]:
# pip install timm
import timm

class ViTCaptcha(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN,
                 model_name='vit_small_patch16_224'):
        super().__init__()
        self.code_len = code_len
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        feat = self.backbone.num_features          # 384 for vit_small
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Dropout(0.2), nn.Linear(feat, num_classes))
            for _ in range(code_len)
        ])

    def forward(self, x):
        f = self.backbone(x)                       # (B, feat) pooled
        return torch.stack([h(f) for h in self.heads], dim=1)  # (B, 6, 33)

# ViT needs 224x224 ImageNet-normalised input.
vit_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

model = ViTCaptcha()
print('Model params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

/home/aryansharma/Desktop/Aryan Pendrive Data/ai_projects/.venv/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
/home/aryansharma/Desktop/Aryan Pendrive Data/ai_projects/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model params: 21741894


In [ ]:
# --- Train Model 4 (run on GPU; ViT benefits from a lower LR) ---
train_loader, val_loader = make_loaders(vit_tf, batch_size=32)
train_classifier(model, train_loader, val_loader, epochs=15, lr=3e-4,
                 ckpt='../Artifacts/model4_vit.pth')

## Model 4b — Advanced ViTCaptcha (column-strip tokens)

A purpose-built Vision Transformer for this captcha layout, instead of a generic 224×224 ViT.

**Tokenisation idea**
1. Drop **10 px from each side** of the width (200 → **180**) to shed the noisy borders.
2. Slice the 180-wide image into **6 vertical strips of 30 × 100 px** (full height, 30 px wide).
3. Treat **each 30×100 strip as a single token** — a linear patch-embedding flattens it to one
   embedding vector. Six strips → six tokens, one per character column.
4. A small **Transformer encoder** (multi-head self-attention) lets the tokens share context
   (helpful for overlapping / smeared characters), then **each token's output decodes to one
   character** via a shared linear head.

Because the six strips line up with the six characters, the natural mapping is **1 token → 1
character**. The head is easily widened to predict 2 characters per token (set
`chars_per_token=2` and use 3 strips) if a layout ever packs two glyphs per column. Output stays
`(B, 6, 33)`, so it plugs straight into `train_classifier` / `evaluate_cls`.

In [15]:
# --- Model 4b: Advanced ViTCaptcha — each vertical 30x100 strip is one token ---
class AdvancedViTCaptcha(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, in_ch=3,
                 img_h=100, img_w=200, side_crop=10, strip_w=30,
                 embed_dim=256, depth=4, n_heads=8, mlp_dim=512,
                 dropout=0.1, chars_per_token=1):
        super().__init__()
        self.side_crop = side_crop
        self.strip_w = strip_w
        self.img_h = img_h
        self.chars_per_token = chars_per_token

        cropped_w = img_w - 2 * side_crop                 # 200 - 20 = 180
        self.num_tokens = cropped_w // strip_w            # 180 / 30 = 6
        assert cropped_w % strip_w == 0, 'cropped width must divide evenly into strips'
        assert self.num_tokens * chars_per_token == code_len, \
            f'{self.num_tokens} tokens x {chars_per_token} char/token must equal {code_len}'

        token_dim = in_ch * img_h * strip_w               # 3 * 100 * 30 = 9000
        self.patch_embed = nn.Linear(token_dim, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_tokens, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.drop = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads, dim_feedforward=mlp_dim,
            dropout=dropout, activation='gelu', batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        # Each token predicts `chars_per_token` characters.
        self.head = nn.Linear(embed_dim, num_classes * chars_per_token)

    def forward(self, x):
        # x: (B, C, H, W=200). Crop the width borders.
        B, C, H, W = x.shape
        x = x[:, :, :, self.side_crop:W - self.side_crop]          # (B, C, H, 180)
        # Split width into non-overlapping 30-px strips -> tokens.
        x = x.unfold(3, self.strip_w, self.strip_w)                # (B, C, H, num_tokens, strip_w)
        x = x.permute(0, 3, 1, 2, 4).contiguous()                  # (B, num_tokens, C, H, strip_w)
        x = x.flatten(2)                                           # (B, num_tokens, C*H*strip_w)

        tok = self.patch_embed(x) + self.pos_embed                 # (B, num_tokens, embed_dim)
        tok = self.drop(tok)
        tok = self.encoder(tok)                                    # (B, num_tokens, embed_dim)
        tok = self.norm(tok)
        out = self.head(tok)                                       # (B, num_tokens, C*classes)
        return out.view(B, self.num_tokens * self.chars_per_token, -1)  # (B, 6, 33)

model = AdvancedViTCaptcha()
_x = torch.randn(2, 3, 100, 200)
print('output shape:', tuple(model(_x).shape))   # expect (2, 6, 33)
print('Model params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

output shape: (2, 6, 33)
Model params: 4423201


In [ ]:
# --- Train Model 4b (run on GPU). Reuses the shared 100x200 ImageNet transform. ---
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model, train_loader, val_loader, epochs=25, lr=5e-4,
                 ckpt='../Artifacts/model4b_advanced_vit.pth')

In [17]:
model = AdvancedViTCaptcha().to(device)
model.load_state_dict(torch.load('../Artifacts/model4b_advanced_vit.pth', map_location=device))
vcer, sacc = evaluate_cls(model, val_loader)
print('Model 4b val CER:', vcer, '| full-code acc:', sacc)

Model 4b val CER: 0.9648157412039353 | full-code acc: 0.0


## Model 5 — Custom Tiny CNN (ultra lightweight, <500k params)

A from-scratch CNN: 4 conv blocks (Conv→BN→ReLU, with pooling) → `AdaptiveAvgPool2d((1, 6))` →
a single shared linear head applied to each of the 6 timesteps → `6 × 33`. Designed for
real-time CPU inference; parameter count is printed and asserted below 500k.

In [18]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN):
        super().__init__()
        self.code_len = code_len
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))
        self.features = nn.Sequential(
            block(3, 32),  nn.MaxPool2d(2),     # 50 x 100
            block(32, 64), nn.MaxPool2d(2),     # 25 x 50
            block(64, 128), nn.MaxPool2d(2),    # 12 x 25
            block(128, 128),                    # keep
        )
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))   # (B, 128, 1, 6)
        self.head = nn.Linear(128, num_classes)           # shared across timesteps

    def forward(self, x):
        f = self.features(x)
        f = self.pool(f).squeeze(2).permute(0, 2, 1)      # (B, 6, 128)
        return self.head(f)                                # (B, 6, 33)

model = TinyCNN()
n6 = sum(p.numel() for p in model.parameters())
print(f'Model 6 params: {n6:,}')
assert n6 < 500_000, 'Tiny CNN exceeded 500k parameter budget'

Model 6 params: 245,793


In [ ]:
# --- Train Model 6 ---
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model, train_loader, val_loader, epochs=25, lr=1e-3,
                 ckpt='../Artifacts/model_tinycnn.pth')

In [20]:
model = TinyCNN().to(device)
model.load_state_dict(torch.load('../Artifacts/model_tinycnn.pth', map_location=device))
vcer, sacc = evaluate_cls(model, val_loader)
print('Model 6 val CER:', vcer, '| full-code acc:', sacc)   

Model 6 val CER: 0.3740203435050859 | full-code acc: 0.02001000500250125


## Model 6 — Hybrid / Ensemble (top-3 models)

Combines the per-position class probabilities of the three best classification models by
**soft-voting** (averaging softmax probabilities), then argmax per position. Soft averaging is
more robust than hard majority voting when models disagree with different confidences. A weighted
variant (weights ∝ each model's validation accuracy) is included.

In [33]:
@torch.no_grad()
def ensemble_predict(models_with_transforms, loader_builder, weights=None):
    """models_with_transforms: list of (model, transform). Each model -> (B, 6, 33).
    Averages softmax probabilities (optionally weighted) across models, position-wise.
    Returns (cer, full_code_acc, predictions)."""
    n = len(models_with_transforms)
    weights = weights or [1.0] * n
    # Build a loader per transform; assumes identical split (same seed) so labels align.
    loaders = [loader_builder(tf)[1] for _, tf in models_with_transforms]  # val loaders
    for m, _ in models_with_transforms:
        m.eval().to(device)

    preds, trues = [], []
    iters = [iter(l) for l in loaders]
    while True:
        try:
            batches = [next(it) for it in iters]
        except StopIteration:
            break
        labels = batches[0][1]
        prob_sum = 0.0
        for w, (m, _), (images, _) in zip(weights, models_with_transforms, batches):
            prob_sum = prob_sum + w * F.softmax(m(images.to(device)), dim=2).cpu()
        idx = prob_sum.argmax(dim=2)
        for pi, ti in zip(idx, labels):
            preds.append(decode(pi)); trues.append(decode(ti))
    seq_acc = np.mean([p == t for p, t in zip(preds, trues)])
    return cer(preds, trues), seq_acc, preds

# Example (after the individual models are trained & loaded):
model1 = CaptchaEfficientNet().to(device)
model1.load_state_dict(torch.load('../Artifacts/model1_efficientnet.pth', map_location=device))
model2 = CRNN_EfficientNet().to(device)
model2.load_state_dict(torch.load('../Artifacts/model2_crnn_bigru.pth', map_location=device))
model3 = CRNN_MobileNet().to(device)
model3.load_state_dict(torch.load('../Artifacts/model3_mobilenet_gru.pth', map_location=device))
combo = [(model2, imagenet_tf), (model1, imagenet_tf), (model3, imagenet_tf)]
e_cer, e_acc, _ = ensemble_predict(combo, make_loaders, weights=[0.25, 0.25, 0.5])
print('Ensemble CER:', e_cer, '| full-code acc:', e_acc)

Ensemble CER: 0.008921127230281808 | full-code acc: 0.9479739869934968


In [21]:
class CRNN_ResNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, code_len=CODE_LEN, hidden=256):
        super().__init__()
        self.code_len = code_len
        backbone = models.resnet18(weights='DEFAULT')
        
        # Freeze all layers except layer4
        for p in backbone.parameters():
            p.requires_grad = False
        for p in backbone.layer4.parameters():
            p.requires_grad = True
        
        # Extract features up to layer4 (remove avgpool and fc)
        self.features = nn.Sequential(*list(backbone.children())[:-2])
        # Output channels: 512 (ResNet18's final conv channels)
        
        self.pool = nn.AdaptiveAvgPool2d((1, code_len))   # -> (B, 512, 1, code_len)
        self.gru = nn.GRU(512, hidden, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        f = self.features(x)                       # (B, 2048, H/32, W/32)
        f = self.pool(f).squeeze(2)                # (B, 2048, code_len)
        f = f.permute(0, 2, 1)                     # (B, code_len, 2048)
        seq, _ = self.gru(f)                       # (B, code_len, 2*hidden)
        return self.fc(seq)                        # (B, code_len, num_classes)

model = CRNN_ResNet()
print('Model params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Model params: 10776097


In [ ]:
# --- Train Model 7 (ResNet18 + GRU) ---
train_loader, val_loader = make_loaders(imagenet_tf)
train_classifier(model, train_loader, val_loader, epochs=25, lr=1e-3,
                 ckpt='../Artifacts/model7_resnet18.pth')

In [24]:
model = CRNN_ResNet().to(device)
model.load_state_dict(torch.load('../Artifacts/model7_resnet18.pth', map_location=device))
vcer, sacc = evaluate_cls(model, val_loader)
print('Model ResNet+GRU val CER:', vcer, '| full-code acc:', sacc)

Model ResNet+GRU val CER: 0.2001000500250125 | full-code acc: 0.24562281140570286


## Comparison 

| Model | Val CER ↓   | Full-code acc ↑   | Params | Inference notes |
|-------|-----------|-----------------|--------|-----------------|
| 1 EfficientNet baseline | 0.09763 | 0.5272 | 5,174,242 | reference |
| 2 CRNN + BiGRU | 0.0727 | 0.6373 | 6,717,757 | expected strongest CNN model |
| 3 MobileNet + GRU | 0.01609 | 0.9069 | 1,774,145 | fastest deep model |
| 4 ViT-Small | 0.9648 | 0.0 | 4,423,201 | data-hungry |
| 5 Tiny CNN | 0.3740 | 0.0200 | 245,793 | real-time CPU |
| 6 Ensemble (0.25+0.25+5) | 0.0089 | 0.9480 | — | usually best CER |
| 7 ResNet | 0.2001 | 0.2456 | 10,776,097 | untrained baseline |
